# 📖 Notebook 4: DynamoDB Streams and CDC

**Change Data Capture (CDC)** is the practice of capturing every change that happens to your data and making it available for downstream processing. DynamoDB has built-in support for this through **DynamoDB Streams**.

## Learning Objectives

By the end of this notebook, you'll understand:
- What DynamoDB Streams are and how they work
- The four stream view types (what data is captured)
- How to enable streams and read change events
- Real-world CDC use cases (search sync, notifications, analytics)
- How streams work under the hood

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 03-technologies/databases/dynamodb
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import boto3
from boto3.dynamodb.conditions import Key
from botocore.exceptions import ClientError
import json
import time

dynamodb = boto3.resource(
    "dynamodb",
    endpoint_url="http://localhost:8000",
    region_name="us-east-1",
    aws_access_key_id="local",
    aws_secret_access_key="local",
)

client = boto3.client(
    "dynamodb",
    endpoint_url="http://localhost:8000",
    region_name="us-east-1",
    aws_access_key_id="local",
    aws_secret_access_key="local",
)

streams_client = boto3.client(
    "dynamodbstreams",
    endpoint_url="http://localhost:8000",
    region_name="us-east-1",
    aws_access_key_id="local",
    aws_secret_access_key="local",
)

try:
    client.list_tables()
    print("✅ Connected to DynamoDB Local")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker-compose up -d")

## 🤔 What Are DynamoDB Streams?

DynamoDB Streams captures a **time-ordered log** of every change to items in a table:
- **INSERT** — a new item was added
- **MODIFY** — an existing item was updated
- **REMOVE** — an item was deleted

Each change is recorded as a **stream record** that downstream applications can read.

### The Analogy

Think of DynamoDB Streams like a security camera for your database:
- The camera records every change that happens (insert, update, delete)
- The footage is stored for 24 hours
- Anyone with access can "replay" the footage to see what happened
- Multiple viewers can watch the same footage independently

### Why is CDC Useful?

| Use Case | How Streams Help |
|----------|------------------|
| **Search sync** | Keep an Elasticsearch index up-to-date with DynamoDB data |
| **Notifications** | Send an email/push when an order status changes |
| **Analytics** | Pipe changes to a data warehouse for reporting |
| **Replication** | Copy data to another region or database |
| **Audit log** | Keep a history of all changes for compliance |

## 📺 Stream View Types

When you enable streams, you choose WHAT data is captured in each stream record:

| View Type | What's Captured | Use When |
|-----------|----------------|----------|
| `KEYS_ONLY` | Just the primary key of the changed item | You only need to know WHICH item changed |
| `NEW_IMAGE` | The complete item AFTER the change | You need the latest state (e.g., sync to search) |
| `OLD_IMAGE` | The complete item BEFORE the change | You need to know what the old value was |
| `NEW_AND_OLD_IMAGES` | Both before and after | You need to compare (e.g., "price changed from X to Y") |

In [ ]:
# Create a table with DynamoDB Streams enabled

TABLE_NAME = "OrdersWithStreams"

try:
    dynamodb.Table(TABLE_NAME).delete()
    dynamodb.Table(TABLE_NAME).wait_until_not_exists()
except ClientError:
    pass

# Create the table with streams enabled
response = client.create_table(
    TableName=TABLE_NAME,
    KeySchema=[
        {"AttributeName": "order_id", "KeyType": "HASH"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "order_id", "AttributeType": "S"},
    ],
    BillingMode="PAY_PER_REQUEST",
    StreamSpecification={
        "StreamEnabled": True,
        "StreamViewType": "NEW_AND_OLD_IMAGES",  # Capture both before and after
    },
)

table = dynamodb.Table(TABLE_NAME)
table.wait_until_exists()

# Get the stream ARN
table_desc = client.describe_table(TableName=TABLE_NAME)["Table"]
stream_arn = table_desc.get("LatestStreamArn", "N/A")

print(f"✅ Created table '{TABLE_NAME}' with streams enabled")
print(f"   Stream View Type: NEW_AND_OLD_IMAGES")
print(f"   Stream ARN: {stream_arn}")
print()
print("💡 Every insert, update, and delete on this table will be recorded in the stream.")

In [ ]:
# Make some changes to generate stream records

print("📝 Making changes to generate stream records...")
print()

# 1. INSERT a new order
table.put_item(Item={
    "order_id": "ORD-100",
    "customer": "Alice",
    "status": "pending",
    "total": 59.99,
})
print("  1️⃣  INSERT: Created order ORD-100 (status: pending, total: $59.99)")

# 2. MODIFY the order status
table.update_item(
    Key={"order_id": "ORD-100"},
    UpdateExpression="SET #s = :s",
    ExpressionAttributeNames={"#s": "status"},
    ExpressionAttributeValues={":s": "shipped"},
)
print("  2️⃣  MODIFY: Updated ORD-100 status to 'shipped'")

# 3. MODIFY again — update status to delivered
table.update_item(
    Key={"order_id": "ORD-100"},
    UpdateExpression="SET #s = :s",
    ExpressionAttributeNames={"#s": "status"},
    ExpressionAttributeValues={":s": "delivered"},
)
print("  3️⃣  MODIFY: Updated ORD-100 status to 'delivered'")

# 4. INSERT another order
table.put_item(Item={
    "order_id": "ORD-101",
    "customer": "Bob",
    "status": "pending",
    "total": 24.99,
})
print("  4️⃣  INSERT: Created order ORD-101")

# 5. REMOVE (delete) the second order
table.delete_item(Key={"order_id": "ORD-101"})
print("  5️⃣  REMOVE: Deleted order ORD-101")

print()
print("✅ Generated 5 stream records (2 inserts, 2 modifies, 1 remove)")

In [ ]:
# Read the stream records
# In production, you'd use AWS Lambda or Kinesis. Here we read directly.

def read_all_stream_records(stream_arn):
    """Read all records from a DynamoDB stream."""
    records = []

    # Get the stream description to find shards
    stream_desc = streams_client.describe_stream(
        StreamArn=stream_arn
    )["StreamDescription"]

    for shard in stream_desc.get("Shards", []):
        # Get a shard iterator (starting point for reading)
        shard_iter = streams_client.get_shard_iterator(
            StreamArn=stream_arn,
            ShardId=shard["ShardId"],
            ShardIteratorType="TRIM_HORIZON",  # Start from the beginning
        )["ShardIterator"]

        # Read records from this shard
        while shard_iter:
            result = streams_client.get_records(
                ShardIterator=shard_iter,
                Limit=100,
            )
            records.extend(result.get("Records", []))
            shard_iter = result.get("NextShardIterator")
            if not result.get("Records"):
                break

    return records

# Read all stream records
stream_records = read_all_stream_records(stream_arn)

print(f"📺 Stream Records ({len(stream_records)} total)")
print("=" * 70)

for i, record in enumerate(stream_records, 1):
    event_name = record["eventName"]
    dynamodb_record = record["dynamodb"]

    # Get the key
    keys = dynamodb_record.get("Keys", {})
    order_id = keys.get("order_id", {}).get("S", "?")

    # Determine what changed
    emoji = {"INSERT": "🆕", "MODIFY": "📝", "REMOVE": "🗑️"}.get(event_name, "?")

    print(f"\n  {emoji} Record {i}: {event_name} on {order_id}")

    if "OldImage" in dynamodb_record:
        old = dynamodb_record["OldImage"]
        old_status = old.get("status", {}).get("S", "N/A")
        print(f"     Old: status={old_status}")

    if "NewImage" in dynamodb_record:
        new = dynamodb_record["NewImage"]
        new_status = new.get("status", {}).get("S", "N/A")
        new_customer = new.get("customer", {}).get("S", "N/A")
        print(f"     New: status={new_status}, customer={new_customer}")

## 🔔 Part 2: Building a CDC Consumer

Let's build a simple CDC consumer that watches for order status changes and "sends" notifications.

In the real world, this would be an **AWS Lambda function** triggered by the stream. Here, we'll simulate it by processing stream records.

In [ ]:
# Simulate a CDC consumer — process stream records and "send" notifications

def process_stream_record(record):
    """Process a single stream record — simulates a Lambda handler."""
    event_name = record["eventName"]
    dynamodb_record = record["dynamodb"]
    keys = dynamodb_record.get("Keys", {})
    order_id = keys.get("order_id", {}).get("S", "?")

    # Use Case 1: Send notification when order status changes
    if event_name == "MODIFY":
        old_image = dynamodb_record.get("OldImage", {})
        new_image = dynamodb_record.get("NewImage", {})

        old_status = old_image.get("status", {}).get("S")
        new_status = new_image.get("status", {}).get("S")
        customer = new_image.get("customer", {}).get("S", "Unknown")

        if old_status != new_status:
            return f"📧 NOTIFY {customer}: Order {order_id} changed from '{old_status}' → '{new_status}'"

    # Use Case 2: Log new orders
    elif event_name == "INSERT":
        new_image = dynamodb_record.get("NewImage", {})
        customer = new_image.get("customer", {}).get("S", "Unknown")
        total = new_image.get("total", {}).get("N", "0")
        return f"📊 ANALYTICS: New order {order_id} from {customer} — ${total}"

    # Use Case 3: Alert on deletions
    elif event_name == "REMOVE":
        old_image = dynamodb_record.get("OldImage", {})
        customer = old_image.get("customer", {}).get("S", "Unknown")
        return f"⚠️  ALERT: Order {order_id} deleted (was for {customer})"

    return None


print("🔔 CDC Consumer Output")
print("=" * 70)

for record in stream_records:
    result = process_stream_record(record)
    if result:
        print(f"  {result}")

print()
print("💡 In production, this logic runs in an AWS Lambda function.")
print("   Lambda is automatically triggered when new stream records appear.")
print("   You could also use Kinesis Data Streams for high-throughput CDC pipelines.")

## 🔍 Part 3: Search Sync (DynamoDB → Elasticsearch)

A common pattern is using DynamoDB Streams to keep a **search index** (like Elasticsearch) in sync with your DynamoDB data.

DynamoDB is great for key-value lookups but terrible for full-text search. By piping changes through Streams, you get the best of both worlds.

Here, we'll simulate this with a simple in-memory search index.

In [ ]:
# Simulate syncing DynamoDB changes to a search index

# In-memory "search index" (simulating Elasticsearch)
search_index = {}

def sync_to_search_index(record):
    """Sync a DynamoDB stream record to the search index."""
    event_name = record["eventName"]
    dynamodb_record = record["dynamodb"]
    keys = dynamodb_record.get("Keys", {})
    order_id = keys.get("order_id", {}).get("S")

    if event_name in ("INSERT", "MODIFY"):
        new_image = dynamodb_record.get("NewImage", {})
        # Convert DynamoDB format to plain dict
        doc = {}
        for attr_name, attr_value in new_image.items():
            for type_key, value in attr_value.items():
                doc[attr_name] = value
        search_index[order_id] = doc
        return f"  📥 Indexed {order_id}: {doc}"

    elif event_name == "REMOVE":
        if order_id in search_index:
            del search_index[order_id]
        return f"  📤 Removed {order_id} from index"


print("🔄 Syncing DynamoDB → Search Index")
print("=" * 70)

for record in stream_records:
    result = sync_to_search_index(record)
    if result:
        print(result)

print()
print("📋 Final Search Index State:")
print("-" * 40)
for order_id, doc in search_index.items():
    print(f"  {order_id}: {doc}")

print()
print("💡 Notice: ORD-101 was inserted then deleted — it's NOT in the final index.")
print("   ORD-100 went through 3 changes — the index has the LATEST state.")
print()
print("   In production, replace the dict with Elasticsearch/OpenSearch calls.")

## 🏗️ Architecture Patterns with DynamoDB Streams

Here are common architectures you'll see (and discuss in interviews):

### Pattern 1: Event-Driven Notifications
```
DynamoDB → Stream → Lambda → SNS/SES → User
```
Example: "Your order has shipped!" email

### Pattern 2: Search Index Sync
```
DynamoDB → Stream → Lambda → Elasticsearch
```
Example: Full-text search on product catalog

### Pattern 3: Real-Time Analytics
```
DynamoDB → Kinesis Data Streams → Kinesis Firehose → S3/Redshift
```
Example: Real-time dashboard of order metrics

### Pattern 4: Cross-Region Replication
```
DynamoDB (us-east-1) → Stream → Lambda → DynamoDB (eu-west-1)
```
Note: For this pattern, use **Global Tables** instead — it's built-in!

### Pattern 5: Audit Trail
```
DynamoDB → Stream → Lambda → Audit Table (append-only)
```
Example: Compliance log of all data changes

## ⚙️ Under the Hood: How Streams Work

1. **Stream records** are organized into **shards** (similar to Kinesis)
2. Each shard contains a sequence of stream records
3. Records are retained for **24 hours** — after that, they're gone
4. Each record has a **sequence number** for ordering
5. Multiple consumers can read the same stream independently
6. Stream records are written **synchronously** with the base table write

### Important Guarantees
- **Exactly once**: Each change appears in the stream exactly once
- **Ordered**: Records within a shard are in the order changes occurred
- **Near real-time**: Records appear within milliseconds of the write
- **No data loss**: The stream record is committed as part of the same transaction as the table write

## 🎯 Key Takeaways

1. **DynamoDB Streams** = built-in CDC that captures every insert, update, and delete
2. Choose your **stream view type** based on what data you need (KEYS_ONLY, NEW_IMAGE, OLD_IMAGE, NEW_AND_OLD_IMAGES)
3. Common use cases: **notifications**, **search sync**, **analytics**, **audit trails**
4. Stream records are retained for **24 hours**
5. In production, use **AWS Lambda** as the stream consumer
6. For high-throughput analytics, pipe through **Kinesis Data Streams** → **Firehose** → **S3/Redshift**

### Interview Tips
- Mention DynamoDB Streams when discussing **cross-store consistency** (e.g., keeping Elasticsearch in sync)
- Mention it for **event-driven architectures** where downstream services react to data changes
- Know that it's **near real-time** but **eventually consistent** — the stream consumer may lag slightly behind
- For cross-region replication, prefer **Global Tables** over building your own stream-based replication

### Series Complete! 🎉
You've now covered all the essential DynamoDB concepts:
1. **Partition keys and sort keys** — how data is organized and queried
2. **Secondary indexes** — querying by non-key attributes
3. **Single-table design** — modeling multiple entities efficiently
4. **Streams and CDC** — reacting to data changes in real time